# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR² dataset, which contains mass spectrometry data from human extracellular vesicles, using the `mlcroissant` Python library.

### Dataset Source
This dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and includes record sets and fields for advanced data analysis.

In [ ]:
# Install mlcroissant if needed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`. The metadata reveals dataset structure, authors, and other descriptive fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
List and review the available record sets, fields, and their `@id`s. All entities are referenced by their `@id` per Croissant recommendations.

In [ ]:
# List all record sets, their @id's and fields
from pprint import pprint

print("Available Record Sets (@id):")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- RecordSet Name: {getattr(record_set, 'name', '')}, @id: {getattr(record_set, '@id', '')}")
    print("  Fields (Columns):")
    for field in getattr(record_set, 'fields', []):
        print(f"    - Field Name: {getattr(field, 'name', '')}, @id: {getattr(field, '@id', '')}, dataType: {getattr(field, 'data_type', '')}")
print(f"\nTotal RecordSets found: {len(record_sets)}")

## 3. Data Extraction
Load data from each record set. Use their `@id`s for all lookups and for later reference. If available, show the first record from each set.

In [ ]:
# Create a mapping from RecordSet @id to the RecordSet object
record_set_map = {getattr(rs, '@id'): rs for rs in dataset.record_sets}
record_set_ids = list(record_set_map.keys())
# If dataset has no record sets, this cell will warn and exit
if not record_set_ids:
    print("No record sets found in dataset - check Croissant metadata.")
else:
    print(f"RecordSet @id's: {record_set_ids}")

# Load all record sets into dataframes for exploration, referencing by @id.
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecordSet: {rs_id}")
    print(f"  Columns: {df.columns.tolist()}")
    display(df.head(1))

## 4. Exploratory Data Analysis (EDA)
Select common numeric fields for inspection, remove outliers, normalize values, and optionally group by a categorical attribute. All fields must be referenced by their `@id`. Adjust `numeric_field_id` and `group_field_id` based on the printed overview.

In [ ]:
# === Edit these according to printouts from previous overview cell ===
# Suppose only one main record set exists; use its @id.
if not record_set_ids:
    print("No record sets available for EDA.")
else:
    record_set_id = record_set_ids[0]  # Use the first record set for illustration
    df = dataframes[record_set_id]
    print(f"Using RecordSet: {record_set_id}")
    # List available fields in this set
    print(f"Fields: {df.columns.tolist()}")
    
    # Try to auto-select a numeric field based on common keywords
    import re
    numeric_candidates = [col for col in df.columns if re.search(r'abundance|count|MW|weight|coverage|pI|score|number|amount|total', col, re.I)]
    
    # If none matched, just pick the first float/integer
    if not numeric_candidates:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_candidates.append(col)
    
    # Choose the first numeric candidate
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # This is a column name == field @id
        print(f"Numeric field selected for analysis: {numeric_field_id}")
        
        # Remove rows where numeric field isn't actually numeric
        df_numeric = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()]
        df_numeric[numeric_field_id] = df_numeric[numeric_field_id].astype(float)
        threshold = df_numeric[numeric_field_id].mean()  # Use mean as example threshold
        filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean value):\nRows: {len(filtered_df)}")
        display(filtered_df.head())
        
        # Normalize the selected field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized field '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Pick a categorical group field
        candidate_groups = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        group_field = candidate_groups[0] if candidate_groups else None
        if group_field:
            print(f"\nGrouping by field: '{group_field}'")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"Grouped mean values by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found for EDA.")

## 5. Visualization
Graph numeric field distribution, and plot mean per group if a group field was selected. 
All fields referenced by their `@id` (column names).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids:
    print("No data available to visualize.")
elif not numeric_candidates:
    print("No numeric field for plotting.")
else:
    # Distribution of the numeric field (filtered)
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field, boxplot or barplot of mean
    if group_field:
        plt.figure(figsize=(12,4))
        # Show only top N group levels for readability
        top_groups = (
            filtered_df[group_field].value_counts().index[:10]
        )
        subset = filtered_df[filtered_df[group_field].isin(top_groups)]
        sns.boxplot(x=group_field, y=numeric_field_id, data=subset)
        plt.title(f"{numeric_field_id} by {group_field} (Top 10 groups)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We loaded and explored the FAIR² Mass Spectrometry dataset with `mlcroissant`.

- The Croissant schema describes dataset entities via their `@id`, referencing each record set and field explicitly.
- Record sets were inspected, data loaded via their `@id`, and key numeric fields were used to demonstrate filtering, normalization, grouping, and visualization.
- This workflow provides a robust, transparent, and reproducible approach for data scientists to analyze FAIR metadata-compliant datasets using open tools.